# BioRob Phase 3 — Manifest + TRUE LOSO Splits

This notebook starts from **Phase 2B labelonly** files only:

`/home/tsultan1/BioRob/Human Subject Data/Sub-*/cleaned/synchronized_proper_lite_union_v3/labelonly/*_icml_consensus_labels.csv`

It automatically skips the 104 files that did not have Phase 1C synchronized output, because those files never reach Phase 2A/2B and therefore do not exist in `labelonly/`.

Important safety choices:
- Does **not** create missing input columns.
- Does **not** modify Phase 2B CSV files.
- Strictly checks the expected 45-column Phase 2B schema.
- Checks `subject_id` against the `Sub-*` folder.
- Checks `task` and `trial` against the filename, e.g., `T515 → task=5, trial=15`.
- Builds TRUE LOSO folds using subject-level split separation.
- Does not balance validation/test data in Phase 3.

In [13]:
# ============================================================
# CELL 1 — Imports and configuration
# ============================================================

from __future__ import annotations

import re
import json
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

# ---------------- PATH CONFIG ----------------
ROOT_DIR = Path("/home/tsultan1/BioRob/Human Subject Data")

# Phase 3 must read only Phase 2B labelonly files.
LABELONLY_SUBPATH = Path("cleaned/synchronized_proper_lite_union_v3/labelonly")
CSV_GLOB = "*_icml_consensus_labels.csv"

# Output dataset folder
DATASET_DIR = ROOT_DIR / "_dataset_icml_v1"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_OUT = DATASET_DIR / "manifest_v1.csv"
SPLITS_OUT = DATASET_DIR / "splits_v1.csv"

AUDIT_DIR = ROOT_DIR / "_audit_phase3_manifest_loso"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- WINDOW CONFIG ----------------
SLIDE_WIN_S = 2.0
SLIDE_STRIDE_S = 0.25
ANCHOR_PRE_S = 2.0
ANCHOR_POST_S = 3.0
ACTIVE_MAJ_FRAC = 0.50

# Phase 2B labelonly files already contain active-only rows for non-T0 trials
# and full REST rows for T0 trials. So sliding windows are the main windows.
ENABLE_ONSET_ANCHOR_WINDOWS = False

# Keep False for clean paper evaluation.
# If balancing is needed, Phase 5 exporter should do train-only balancing later.
BALANCE_TRAIN_WINDOWS_IN_PHASE3 = False

# TRUE LOSO validation-subject selection
VAL_SELECTION_MODE = "smallest_other"  # options: "smallest_other", "fixed"
VAL_SUBJECT_ID = 1

# Strict schema check.
# True = skip files with missing OR extra columns.
STRICT_EXACT_INPUT_SCHEMA = True

# Optional MD5 for manifest files. Usually False to save time.
WRITE_FILE_MD5 = False

print("ROOT_DIR:", ROOT_DIR)
print("LABELONLY_SUBPATH:", LABELONLY_SUBPATH)
print("DATASET_DIR:", DATASET_DIR)
print("STRICT_EXACT_INPUT_SCHEMA:", STRICT_EXACT_INPUT_SCHEMA)

ROOT_DIR: /home/tsultan1/BioRob/Human Subject Data
LABELONLY_SUBPATH: cleaned/synchronized_proper_lite_union_v3/labelonly
DATASET_DIR: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1
STRICT_EXACT_INPUT_SCHEMA: True


In [14]:
# ============================================================
# CELL 2 — Exact Phase 2B input schema
# This list must match the Phase 2B labelonly CSV columns.
# Do not add columns here unless Phase 2B truly outputs them.
# ============================================================

EXPECTED_PHASE2B_COLUMNS = [
    "subject_id", "task", "trial", "Timestamp_seconds", "Timestamp_ms",
    "EMG_Ch1", "EMG_Ch2", "EMG_Ch3", "EMG_Ch4",
    "EEG_Ch1", "EEG_Ch2", "EEG_Ch3", "EEG_Ch4",
    "EEG_Ch5", "EEG_Ch6", "EEG_Ch7", "EEG_Ch8",
    "ET_GazeLeftx", "ET_GazeLefty", "ET_GazeRightx", "ET_GazeRighty",
    "ET_PupilLeft", "ET_PupilRight",
    "ET_ValidityLeftEye", "ET_ValidityRightEye",
    "ET_Blink", "ET_Fixation", "ET_Worn",
    "ET_DistanceLeft", "ET_DistanceRight",
    "ET_GyroX", "ET_GyroY", "ET_GyroZ",
    "ET_AccX", "ET_AccY", "ET_AccZ",
    "ET_HeadRotationPitch", "ET_HeadRotationYaw", "ET_HeadRotationRoll",
    "active_raw", "active", "active_prob",
    "label_action", "label_11", "task_target",
]

REQUIRED_FOR_PHASE3 = [
    "Timestamp_seconds",
    "active",
    "task",
    "trial",
    "subject_id",
    "label_action",
    "task_target",
]

print("Expected Phase 2B columns:", len(EXPECTED_PHASE2B_COLUMNS))
print("Required Phase 3 columns :", REQUIRED_FOR_PHASE3)

Expected Phase 2B columns: 45
Required Phase 3 columns : ['Timestamp_seconds', 'active', 'task', 'trial', 'subject_id', 'label_action', 'task_target']


In [15]:
# ============================================================
# CELL 3 — Helper functions
# ============================================================

def subject_sort_key(path: Path) -> int:
    m = re.search(r"Sub-(\d+)", path.name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else 10**9


def parse_subject_from_path(path: Path) -> int | None:
    for part in path.parts:
        m = re.fullmatch(r"Sub-(\d+)", part, flags=re.IGNORECASE)
        if m:
            return int(m.group(1))
    return None


def parse_task_trial_from_filename(path: Path) -> tuple[int | None, int | None, str | None]:
    """
    Parses task/trial from names such as:
    003_T515_synchronized_corrected_icml_consensus_labels.csv
    001_T03_synchronized_corrected_icml_consensus_labels.csv

    Convention:
      first digit after T/M = task
      remaining digits = trial
    Examples:
      T515 -> task=5, trial=15
      T03  -> task=0, trial=3
      T112 -> task=1, trial=12
    """
    m = re.search(r"_(T|M)(\d+)", path.name, flags=re.IGNORECASE)
    if not m:
        return None, None, None

    mode = m.group(1).upper()
    digits = m.group(2)

    if len(digits) < 2:
        return None, None, mode

    task = int(digits[0])
    trial = int(digits[1:])
    return task, trial, mode


def median_fs_from_time(t: np.ndarray) -> float:
    t = np.asarray(t, dtype=float)
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        return np.nan
    return float(1.0 / np.median(dt))


def file_md5(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as fh:
        while True:
            b = fh.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def to_int_scalar(series_or_value, default=None):
    try:
        if hasattr(series_or_value, "dropna"):
            vals = series_or_value.dropna().unique()
            if len(vals) != 1:
                return default
            x = vals[0]
        else:
            x = series_or_value
        if pd.isna(x):
            return default
        return int(float(x))
    except Exception:
        return default


def safe_mode_int(values, default=0, ignore_nan=True):
    arr = pd.to_numeric(pd.Series(values), errors="coerce")
    if ignore_nan:
        arr = arr.dropna()
    if len(arr) == 0:
        return int(default)
    counts = Counter(arr.astype(int).tolist())
    # Stable tie-break: count desc, class asc
    return int(sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[0][0])


def collect_labelonly_files() -> list[Path]:
    files = []

    sub_dirs = sorted(
        [p for p in ROOT_DIR.glob("Sub-*") if p.is_dir()],
        key=subject_sort_key,
    )

    print("Found Sub-* folders:", len(sub_dirs))

    for sub_dir in sub_dirs:
        labelonly_dir = sub_dir / LABELONLY_SUBPATH

        if not labelonly_dir.exists():
            print(f"[skip] No labelonly folder: {labelonly_dir}")
            continue

        got = sorted(labelonly_dir.glob(CSV_GLOB))

        if len(got) == 0:
            print(f"[skip] No labelonly CSVs: {labelonly_dir}")
            continue

        print(f"[use] {sub_dir.name}: {len(got)} labelonly files")
        files.extend(got)

    return files


def choose_val_subject(subjects: list[int], test_subject: int) -> int:
    others = sorted([s for s in subjects if s != test_subject])

    if len(others) == 0:
        return test_subject

    if VAL_SELECTION_MODE == "fixed":
        if VAL_SUBJECT_ID in others:
            return int(VAL_SUBJECT_ID)

    return int(others[0])


def validate_input_file(path: Path, full_check: bool = True) -> tuple[bool, dict]:
    """
    Checks a Phase 2B labelonly CSV without creating or modifying columns.
    """
    info = {
        "file": str(path),
        "subject_folder": parse_subject_from_path(path),
        "filename_task": None,
        "filename_trial": None,
        "filename_mode": None,
        "csv_subject_id": None,
        "csv_task": None,
        "csv_trial": None,
        "n_rows": None,
        "n_cols": None,
        "fs_hz": np.nan,
        "duration_s": np.nan,
        "status": "OK",
        "reason": "",
    }

    f_task, f_trial, f_mode = parse_task_trial_from_filename(path)
    info["filename_task"] = f_task
    info["filename_trial"] = f_trial
    info["filename_mode"] = f_mode

    try:
        if full_check:
            df = pd.read_csv(path, low_memory=False)
        else:
            df = pd.read_csv(path, low_memory=False, nrows=20)

        info["n_rows"] = int(len(df))
        info["n_cols"] = int(len(df.columns))

        cols = list(df.columns)
        missing = [c for c in EXPECTED_PHASE2B_COLUMNS if c not in cols]
        extra = [c for c in cols if c not in EXPECTED_PHASE2B_COLUMNS]

        if missing:
            info["status"] = "SKIP"
            info["reason"] = f"Missing columns: {missing}"
            return False, info

        if STRICT_EXACT_INPUT_SCHEMA and extra:
            info["status"] = "SKIP"
            info["reason"] = f"Extra columns: {extra}"
            return False, info

        # Required columns check
        missing_req = [c for c in REQUIRED_FOR_PHASE3 if c not in cols]
        if missing_req:
            info["status"] = "SKIP"
            info["reason"] = f"Missing Phase3 required columns: {missing_req}"
            return False, info

        if full_check and len(df) < 2:
            info["status"] = "SKIP"
            info["reason"] = "Too few rows"
            return False, info

        csv_subject = to_int_scalar(df["subject_id"])
        csv_task = to_int_scalar(df["task"])
        csv_trial = to_int_scalar(df["trial"])

        info["csv_subject_id"] = csv_subject
        info["csv_task"] = csv_task
        info["csv_trial"] = csv_trial

        if csv_subject is None:
            info["status"] = "SKIP"
            info["reason"] = "subject_id is missing or not unique"
            return False, info

        if csv_task is None:
            info["status"] = "SKIP"
            info["reason"] = "task is missing or not unique"
            return False, info

        if csv_trial is None:
            info["status"] = "SKIP"
            info["reason"] = "trial is missing or not unique"
            return False, info

        if info["subject_folder"] is not None and csv_subject != info["subject_folder"]:
            info["status"] = "SKIP"
            info["reason"] = f"subject_id mismatch: CSV={csv_subject}, folder=Sub-{info['subject_folder']}"
            return False, info

        if f_task is not None and csv_task != f_task:
            info["status"] = "SKIP"
            info["reason"] = f"task mismatch: CSV={csv_task}, filename={f_task}"
            return False, info

        if f_trial is not None and csv_trial != f_trial:
            info["status"] = "SKIP"
            info["reason"] = f"trial mismatch: CSV={csv_trial}, filename={f_trial}"
            return False, info

        if full_check:
            t = pd.to_numeric(df["Timestamp_seconds"], errors="coerce").to_numpy(dtype=float)
            fs = median_fs_from_time(t)
            dur = float(np.nanmax(t) - np.nanmin(t)) if len(t) else np.nan
            info["fs_hz"] = fs
            info["duration_s"] = dur

            if not np.isfinite(fs) or fs <= 0:
                info["status"] = "SKIP"
                info["reason"] = "Invalid sampling rate from Timestamp_seconds"
                return False, info

            if abs(fs - 250.0) > 5.0:
                info["status"] = "SKIP"
                info["reason"] = f"Unexpected fs_hz={fs:.3f}"
                return False, info

        return True, info

    except Exception as e:
        info["status"] = "SKIP"
        info["reason"] = f"Read/validation error: {e}"
        return False, info

In [16]:
# ============================================================
# CELL 4 — Collect Phase 2B labelonly files and audit inputs
# This automatically skips the 104 missing Phase 1C files because they
# do not have Phase 2B labelonly outputs.
# ============================================================

labelonly_files_all = collect_labelonly_files()

print("\nTotal discovered Phase 2B labelonly files:", len(labelonly_files_all))

# If the Phase 1C exclusion report exists, print it for traceability.
phase1c_exclusion_report = ROOT_DIR / "_audit_phase1c_exclusions" / "excluded_missing_phase1c_files.csv"
if phase1c_exclusion_report.exists():
    excluded_df = pd.read_csv(phase1c_exclusion_report)
    print("\nPhase 1C missing/excluded report found:")
    print(phase1c_exclusion_report)
    print("Excluded missing Phase 1C files:", len(excluded_df))
else:
    print("\nNo Phase 1C exclusion report found. This is okay if you did not save it.")

audit_rows = []
valid_files = []

for i, path in enumerate(labelonly_files_all, start=1):
    ok, info = validate_input_file(path, full_check=True)
    audit_rows.append(info)
    if ok:
        valid_files.append(path)

input_audit = pd.DataFrame(audit_rows)

audit_path = AUDIT_DIR / "phase3_labelonly_input_audit.csv"
input_audit.to_csv(audit_path, index=False)

print("\n" + "=" * 100)
print("PHASE 3 INPUT AUDIT SUMMARY")
print("=" * 100)
print(input_audit["status"].value_counts(dropna=False))
print("Valid files:", len(valid_files))
print("Skipped files:", int((input_audit["status"] != "OK").sum()))
print("Saved audit:", audit_path)

print("\nFiles by subject:")
if len(input_audit) > 0:
    display(
        input_audit[input_audit["status"] == "OK"]
        .groupby("csv_subject_id")["file"]
        .count()
        .reset_index(name="valid_labelonly_files")
    )

if len(valid_files) == 0:
    raise RuntimeError("No valid Phase 2B labelonly files found. Stop before Phase 3.")

# Expected from your current pipeline: 1552 if Phase 2B has completed for all usable Phase 1C files.
if len(valid_files) == 1552:
    print("\n✅ Found 1552 valid labelonly files. The 104 missing Phase 1C files are skipped.")
else:
    print(f"\n⚠️ Found {len(valid_files)} valid labelonly files, not 1552.")
    print("If Phase 2B is still running or incomplete, finish Phase 2B before final Phase 3.")

Found Sub-* folders: 20
[use] Sub-1: 83 labelonly files
[use] Sub-2: 83 labelonly files
[use] Sub-3: 82 labelonly files
[use] Sub-5: 83 labelonly files
[use] Sub-6: 81 labelonly files
[use] Sub-7: 82 labelonly files
[use] Sub-8: 80 labelonly files
[use] Sub-9: 82 labelonly files
[use] Sub-10: 82 labelonly files
[use] Sub-11: 80 labelonly files
[use] Sub-12: 81 labelonly files
[skip] No labelonly folder: /home/tsultan1/BioRob/Human Subject Data/Sub-13/cleaned/synchronized_proper_lite_union_v3/labelonly
[use] Sub-14: 82 labelonly files
[use] Sub-16: 80 labelonly files
[use] Sub-17: 83 labelonly files
[use] Sub-19: 83 labelonly files
[use] Sub-20: 83 labelonly files
[use] Sub-21: 81 labelonly files
[use] Sub-22: 81 labelonly files
[use] Sub-23: 80 labelonly files

Total discovered Phase 2B labelonly files: 1552

Phase 1C missing/excluded report found:
/home/tsultan1/BioRob/Human Subject Data/_audit_phase1c_exclusions/excluded_missing_phase1c_files.csv
Excluded missing Phase 1C files: 104


,csv_subject_id,valid_labelonly_files
0,1,83
1,2,83
2,3,82
3,5,83
4,6,81
5,7,82
6,8,80
7,9,82
8,10,82
9,11,80



✅ Found 1552 valid labelonly files. The 104 missing Phase 1C files are skipped.


In [17]:
# ============================================================
# CELL 5 — Inspect one Phase 2B labelonly CSV
# This is a safety check only. It does not modify the data.
# ============================================================

sample_file = valid_files[0]
print("Sample file:", sample_file)

sample_df = pd.read_csv(sample_file, low_memory=False)

print("Shape:", sample_df.shape)
print("\nColumns:")
for c in sample_df.columns:
    print(" -", c)

print("\nMetadata:")
print("subject_id:", sample_df["subject_id"].dropna().unique())
print("task      :", sample_df["task"].dropna().unique())
print("trial     :", sample_df["trial"].dropna().unique())

print("\nLabel columns summary:")
for c in ["active", "label_action", "label_11", "task_target"]:
    print(f"\n{c}:")
    print(sample_df[c].value_counts(dropna=False).sort_index())

print("\nTimestamp check:")
t = pd.to_numeric(sample_df["Timestamp_seconds"], errors="coerce").to_numpy(dtype=float)
print("median fs:", median_fs_from_time(t))
print("duration :", float(np.nanmax(t) - np.nanmin(t)))

print("\n✅ Sample Phase 2B file inspection complete.")

Sample file: /home/tsultan1/BioRob/Human Subject Data/Sub-1/cleaned/synchronized_proper_lite_union_v3/labelonly/001_T03_synchronized_corrected_icml_consensus_labels.csv
Shape: (3059, 45)

Columns:
 - subject_id
 - task
 - trial
 - Timestamp_seconds
 - Timestamp_ms
 - EMG_Ch1
 - EMG_Ch2
 - EMG_Ch3
 - EMG_Ch4
 - EEG_Ch1
 - EEG_Ch2
 - EEG_Ch3
 - EEG_Ch4
 - EEG_Ch5
 - EEG_Ch6
 - EEG_Ch7
 - EEG_Ch8
 - ET_GazeLeftx
 - ET_GazeLefty
 - ET_GazeRightx
 - ET_GazeRighty
 - ET_PupilLeft
 - ET_PupilRight
 - ET_ValidityLeftEye
 - ET_ValidityRightEye
 - ET_Blink
 - ET_Fixation
 - ET_Worn
 - ET_DistanceLeft
 - ET_DistanceRight
 - ET_GyroX
 - ET_GyroY
 - ET_GyroZ
 - ET_AccX
 - ET_AccY
 - ET_AccZ
 - ET_HeadRotationPitch
 - ET_HeadRotationYaw
 - ET_HeadRotationRoll
 - active_raw
 - active
 - active_prob
 - label_action
 - label_11
 - task_target

Metadata:
subject_id: [1]
task      : [0]
trial     : [3]

Label columns summary:

active:
active
0    3059
Name: count, dtype: int64

label_action:
label_action

In [18]:
# ============================================================
# CELL 6 — Build manifest_v1.csv
# One row per valid Phase 2B labelonly file
# ============================================================

manifest_rows = []
manifest_skipped = []

for path in valid_files:
    try:
        df = pd.read_csv(path, low_memory=False)

        # Validate again before adding to manifest.
        ok, info = validate_input_file(path, full_check=True)
        if not ok:
            manifest_skipped.append({"file": str(path), "reason": info["reason"]})
            continue

        t = pd.to_numeric(df["Timestamp_seconds"], errors="coerce").to_numpy(dtype=float)
        fs = median_fs_from_time(t)
        duration_s = float(np.nanmax(t) - np.nanmin(t)) if len(t) else np.nan

        subj = int(df["subject_id"].dropna().unique()[0])
        task = int(df["task"].dropna().unique()[0])
        trial = int(df["trial"].dropna().unique()[0])

        row = {
            "file": str(path.resolve()),
            "subject_id": subj,
            "fs_hz": round(float(fs), 6) if np.isfinite(fs) else np.nan,
            "duration_s": round(float(duration_s), 6) if np.isfinite(duration_s) else np.nan,
            "task_code": task,
            "trial_id": trial,
            "fold_id": subj,
        }

        if WRITE_FILE_MD5:
            row["md5"] = file_md5(path)

        manifest_rows.append(row)

    except Exception as e:
        manifest_skipped.append({"file": str(path), "reason": str(e)})

manifest = pd.DataFrame(manifest_rows)

if len(manifest) == 0:
    raise RuntimeError("Manifest is empty. Stop.")

manifest = manifest.sort_values(["subject_id", "task_code", "trial_id", "file"]).reset_index(drop=True)

manifest.to_csv(MANIFEST_OUT, index=False)

print("=" * 100)
print("MANIFEST CREATED")
print("=" * 100)
print("Saved:", MANIFEST_OUT)
print("Rows :", len(manifest))
print("Subjects:", sorted(manifest["subject_id"].unique().tolist()))

print("\nManifest by subject:")
display(manifest.groupby("subject_id")["file"].count().reset_index(name="files"))

if manifest_skipped:
    skipped_path = DATASET_DIR / "manifest_v1_skipped.csv"
    pd.DataFrame(manifest_skipped).to_csv(skipped_path, index=False)
    print("\n⚠️ Manifest skipped files saved:", skipped_path)
else:
    print("\n✅ No manifest files skipped after input audit.")

MANIFEST CREATED
Saved: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/manifest_v1.csv
Rows : 1552
Subjects: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]

Manifest by subject:


,subject_id,files
0,1,83
1,2,83
2,3,82
3,5,83
4,6,81
5,7,82
6,8,80
7,9,82
8,10,82
9,11,80



✅ No manifest files skipped after input audit.


In [19]:
# ============================================================
# CELL 7 — Windowing functions for Phase 2B labelonly files
# ============================================================

def onsets_from_binary(binary_arr: np.ndarray) -> np.ndarray:
    x = (np.asarray(binary_arr).astype(int) > 0).astype(int)
    d = np.diff(np.r_[0, x])
    return np.where(d == 1)[0]


def label_window_from_columns(df_window: pd.DataFrame) -> tuple[int, int]:
    """
    Returns (label_action, task_target) for one window.
    Uses existing Phase 2B label columns only.
    Does not create any new input columns.
    """
    label_values = pd.to_numeric(df_window["label_action"], errors="coerce").fillna(0).to_numpy(dtype=float)
    action_frac = float(np.mean(label_values >= 0.5)) if len(label_values) else 0.0

    label_action = 1 if action_frac >= ACTIVE_MAJ_FRAC else 0

    if label_action == 1:
        task_values = pd.to_numeric(df_window["task_target"], errors="coerce").fillna(0).astype(int).to_numpy()
        nonzero = task_values[task_values != 0]
        task_target = safe_mode_int(nonzero, default=0) if len(nonzero) else 0
    else:
        task_target = 0

    return int(label_action), int(task_target)


def build_windows_for_file(path: Path, df: pd.DataFrame, fs: float, fold_id: int, split_tag: str) -> tuple[list[dict], list[dict]]:
    """
    Build sliding and optional onset-anchor windows.
    For Phase 2B labelonly files:
      - non-T0 files usually contain only active rows
      - T0 files contain REST rows
    """
    rows = []
    skipped = []

    n = len(df)
    if n < 2:
        skipped.append({"file": str(path), "reason": "too_few_rows", "n_rows": n})
        return rows, skipped

    subj = int(df["subject_id"].dropna().unique()[0])
    task_code = int(df["task"].dropna().unique()[0])
    trial_id = int(df["trial"].dropna().unique()[0])

    win_n = max(1, int(round(SLIDE_WIN_S * fs)))
    stride_n = max(1, int(round(SLIDE_STRIDE_S * fs)))
    pre_n = max(0, int(round(ANCHOR_PRE_S * fs)))
    post_n = max(1, int(round(ANCHOR_POST_S * fs)))

    if n < win_n:
        skipped.append({
            "file": str(path),
            "reason": f"shorter_than_window_{SLIDE_WIN_S}s",
            "n_rows": int(n),
            "win_n": int(win_n),
            "fs": float(fs),
        })
        return rows, skipped

    # Sliding windows
    for s in range(0, n - win_n + 1, stride_n):
        e = s + win_n
        dfw = df.iloc[s:e]

        label_action, task_target = label_window_from_columns(dfw)

        rows.append({
            "file": str(path.resolve()),
            "subject_id": subj,
            "split": split_tag,
            "type": "sliding",
            "task_target": int(task_target),
            "label_action": int(label_action),
            "start_idx": int(s),
            "end_idx": int(e),
            "task_code": int(task_code),
            "trial_id": int(trial_id),
            "fold_id": int(fold_id),
        })

    # Optional onset-anchor windows.
    # For active-only labelonly files, the onset is often at index 0;
    # those windows are skipped if pre-window would go negative.
    if ENABLE_ONSET_ANCHOR_WINDOWS:
        label_arr = pd.to_numeric(df["label_action"], errors="coerce").fillna(0).astype(int).to_numpy()
        onsets = onsets_from_binary(label_arr)

        for o in onsets:
            s = int(o - pre_n)
            e = int(o + post_n)

            if s < 0 or e > n or e <= s:
                continue

            dfw = df.iloc[s:e]
            label_action, task_target = label_window_from_columns(dfw)

            # Onset-anchor should represent action onset.
            if label_action != 1:
                continue

            rows.append({
                "file": str(path.resolve()),
                "subject_id": subj,
                "split": split_tag,
                "type": "onset_anchor",
                "task_target": int(task_target),
                "label_action": 1,
                "start_idx": int(s),
                "end_idx": int(e),
                "task_code": int(task_code),
                "trial_id": int(trial_id),
                "fold_id": int(fold_id),
            })

    return rows, skipped

In [20]:
# ============================================================
# CELL 8 — Build TRUE LOSO splits_v1.csv
# Subject-level split: one subject test, one different subject val,
# all remaining subjects train.
# ============================================================

subjects = sorted(manifest["subject_id"].unique().astype(int).tolist())

print("=" * 100)
print("TRUE LOSO SETUP")
print("=" * 100)
print("Subjects:", subjects)
print("Number of subjects:", len(subjects))

if len(subjects) < 3:
    raise RuntimeError("Need at least 3 subjects for train/val/test LOSO.")

splits_rows = []
window_skipped_rows = []

# Cache loaded files by path to avoid re-reading too often.
# If memory becomes a problem, set CACHE_FILES = False.
CACHE_FILES = False
df_cache = {}

for test_subject in subjects:
    val_subject = choose_val_subject(subjects, test_subject)

    split_for_subj = {}
    for s in subjects:
        if s == test_subject:
            split_for_subj[s] = "test"
        elif s == val_subject:
            split_for_subj[s] = "val"
        else:
            split_for_subj[s] = "train"

    train_subjects = [s for s in subjects if split_for_subj[s] == "train"]

    print("\n" + "=" * 100)
    print(f"LOSO fold_id={test_subject}")
    print("Train subjects:", train_subjects)
    print("Val subject   :", val_subject)
    print("Test subject  :", test_subject)

    for row in manifest.itertuples(index=False):
        path = Path(row.file)
        subj = int(row.subject_id)
        split_tag = split_for_subj[subj]

        try:
            if CACHE_FILES and str(path) in df_cache:
                df = df_cache[str(path)]
            else:
                df = pd.read_csv(path, low_memory=False)
                if CACHE_FILES:
                    df_cache[str(path)] = df

            ok, info = validate_input_file(path, full_check=False)
            if not ok:
                window_skipped_rows.append({
                    "file": str(path),
                    "fold_id": test_subject,
                    "split": split_tag,
                    "reason": f"validation_failed: {info['reason']}",
                })
                continue

            t = pd.to_numeric(df["Timestamp_seconds"], errors="coerce").to_numpy(dtype=float)
            fs = median_fs_from_time(t)

            if not np.isfinite(fs) or fs <= 0:
                window_skipped_rows.append({
                    "file": str(path),
                    "fold_id": test_subject,
                    "split": split_tag,
                    "reason": f"bad_fs: {fs}",
                })
                continue

            rows, skipped = build_windows_for_file(
                path=path,
                df=df,
                fs=fs,
                fold_id=int(test_subject),
                split_tag=split_tag,
            )

            splits_rows.extend(rows)

            for sk in skipped:
                sk["fold_id"] = int(test_subject)
                sk["split"] = split_tag
                window_skipped_rows.append(sk)

        except Exception as e:
            window_skipped_rows.append({
                "file": str(path),
                "fold_id": int(test_subject),
                "split": split_tag,
                "reason": f"windowing_exception: {e}",
            })

splits = pd.DataFrame(splits_rows)

if len(splits) == 0:
    raise RuntimeError("No windows were created. Check Phase 2B labelonly lengths and window settings.")

ordered_cols = [
    "file", "subject_id", "split", "type", "task_target", "label_action",
    "start_idx", "end_idx", "task_code", "trial_id", "fold_id",
]

splits = splits[ordered_cols].sort_values(
    ["fold_id", "split", "subject_id", "task_code", "trial_id", "file", "start_idx"]
).reset_index(drop=True)

print("\n" + "=" * 100)
print("WINDOWING COMPLETE")
print("=" * 100)
print("Total windows:", len(splits))
print("Window types:")
print(splits["type"].value_counts(dropna=False))
print("\nAction labels:")
print(splits["label_action"].value_counts(dropna=False).sort_index())
print("\nTask labels:")
print(splits["task_target"].value_counts(dropna=False).sort_index())

if window_skipped_rows:
    window_skipped_df = pd.DataFrame(window_skipped_rows)
    skipped_path = AUDIT_DIR / "phase3_windowing_skipped.csv"
    window_skipped_df.to_csv(skipped_path, index=False)
    print("\n⚠️ Windowing skipped rows saved:", skipped_path)
    print("Skipped count:", len(window_skipped_df))
else:
    print("\n✅ No files/windows skipped during windowing.")

TRUE LOSO SETUP
Subjects: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Number of subjects: 19

LOSO fold_id=1
Train subjects: [3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Val subject   : 2
Test subject  : 1

LOSO fold_id=2
Train subjects: [3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Val subject   : 1
Test subject  : 2

LOSO fold_id=3
Train subjects: [2, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Val subject   : 1
Test subject  : 3

LOSO fold_id=5
Train subjects: [2, 3, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Val subject   : 1
Test subject  : 5

LOSO fold_id=6
Train subjects: [2, 3, 5, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Val subject   : 1
Test subject  : 6

LOSO fold_id=7
Train subjects: [2, 3, 5, 6, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Val subject   : 1
Test subject  : 7

LOSO fold_id=8
Train subjects: [2, 3, 5, 6, 7, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
V

In [21]:
# ============================================================
# CELL 9 — Optional train-only balancing inside Phase 3
# Default is OFF. Keep OFF for the cleanest evaluation protocol.
# Phase 5 can do train-only balancing later.
# ============================================================

def balance_train_only_per_fold(splits_df: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    """
    Optional: balance only TRAIN rows inside each fold.
    Val/test rows are never changed.
    This is disabled by default.
    """
    out_parts = []

    for fold_id, fold_df in splits_df.groupby("fold_id", sort=True):
        keep_parts = []

        # Keep val/test unchanged
        keep_parts.append(fold_df[fold_df["split"] != "train"])

        train_df = fold_df[fold_df["split"] == "train"]

        if len(train_df) == 0:
            out_parts.append(fold_df)
            continue

        counts = train_df["label_action"].value_counts()
        if len(counts) < 2:
            keep_parts.append(train_df)
            out_parts.append(pd.concat(keep_parts, ignore_index=True))
            continue

        target_n = int(counts.min())

        sampled_parts = []
        for cls, g in train_df.groupby("label_action"):
            if len(g) > target_n:
                sampled_parts.append(g.sample(n=target_n, random_state=random_state))
            else:
                sampled_parts.append(g)

        keep_parts.append(pd.concat(sampled_parts, ignore_index=True))
        out_parts.append(pd.concat(keep_parts, ignore_index=True))

    return pd.concat(out_parts, ignore_index=True)


if BALANCE_TRAIN_WINDOWS_IN_PHASE3:
    print("⚠️ Applying train-only balancing in Phase 3.")
    splits_final = balance_train_only_per_fold(splits)
else:
    print("✅ Phase 3 balancing is OFF. Splits are unbalanced/raw windows.")
    splits_final = splits.copy()

splits_final = splits_final[ordered_cols].sort_values(
    ["fold_id", "split", "subject_id", "task_code", "trial_id", "file", "start_idx"]
).reset_index(drop=True)

print("Final split rows:", len(splits_final))
print("\nFinal label distribution:")
print(splits_final["label_action"].value_counts(dropna=False).sort_index())

✅ Phase 3 balancing is OFF. Splits are unbalanced/raw windows.
Final split rows: 475798

Final label distribution:
label_action
0     48222
1    427576
Name: count, dtype: int64


In [22]:
# ============================================================
# CELL 10 — Leakage checks and distribution reports
# ============================================================

leakage_rows = []

for fold_id, g in splits_final.groupby("fold_id", sort=True):
    train_subjects = set(g.loc[g["split"] == "train", "subject_id"].astype(int).unique().tolist())
    val_subjects = set(g.loc[g["split"] == "val", "subject_id"].astype(int).unique().tolist())
    test_subjects = set(g.loc[g["split"] == "test", "subject_id"].astype(int).unique().tolist())

    intersections = {
        "train_val": sorted(train_subjects & val_subjects),
        "train_test": sorted(train_subjects & test_subjects),
        "val_test": sorted(val_subjects & test_subjects),
    }

    if any(len(v) > 0 for v in intersections.values()):
        leakage_rows.append({
            "fold_id": fold_id,
            "train_val_overlap": intersections["train_val"],
            "train_test_overlap": intersections["train_test"],
            "val_test_overlap": intersections["val_test"],
        })

if leakage_rows:
    leakage_df = pd.DataFrame(leakage_rows)
    leakage_path = AUDIT_DIR / "phase3_leakage_errors.csv"
    leakage_df.to_csv(leakage_path, index=False)
    print("❌ LEAKAGE DETECTED. Saved:", leakage_path)
    display(leakage_df)
    raise RuntimeError("Subject leakage detected in LOSO splits.")
else:
    print("✅ No subject leakage detected across train/val/test within any fold.")

fold_summary = (
    splits_final
    .groupby(["fold_id", "split"])
    .agg(
        n_windows=("file", "count"),
        n_subjects=("subject_id", "nunique"),
        n_files=("file", "nunique"),
        action_pos=("label_action", "sum"),
        action_mean=("label_action", "mean"),
    )
    .reset_index()
)

fold_summary_path = AUDIT_DIR / "phase3_fold_split_summary.csv"
fold_summary.to_csv(fold_summary_path, index=False)

print("\nFold/split summary saved:", fold_summary_path)
display(fold_summary.head(20))

task_dist = (
    splits_final
    .groupby(["fold_id", "split", "task_target"])
    .size()
    .reset_index(name="n_windows")
)

task_dist_path = AUDIT_DIR / "phase3_task_distribution_by_fold_split.csv"
task_dist.to_csv(task_dist_path, index=False)

print("\nTask distribution saved:", task_dist_path)
display(task_dist.head(30))

✅ No subject leakage detected across train/val/test within any fold.

Fold/split summary saved: /home/tsultan1/BioRob/Human Subject Data/_audit_phase3_manifest_loso/phase3_fold_split_summary.csv


,fold_id,split,n_windows,n_subjects,n_files,action_pos,action_mean
0,1,test,2068,1,83,1927,0.931818
1,1,train,20467,17,1384,18302,0.894220
2,1,val,2507,1,83,2275,0.907459
3,2,test,2507,1,83,2275,0.907459
4,2,train,20467,17,1384,18302,0.894220
5,2,val,2068,1,83,1927,0.931818
6,3,test,2430,1,82,2295,0.944444
7,3,train,20544,17,1385,18282,0.889895
8,3,val,2068,1,83,1927,0.931818
9,5,test,1377,1,83,1260,0.915033



Task distribution saved: /home/tsultan1/BioRob/Human Subject Data/_audit_phase3_manifest_loso/phase3_task_distribution_by_fold_split.csv


,fold_id,split,task_target,n_windows
0,1,test,0,141
1,1,test,1,411
2,1,test,2,535
3,1,test,3,343
4,1,test,4,400
5,1,test,5,238
6,1,train,0,2165
7,1,train,1,3950
8,1,train,2,4917
9,1,train,3,3374


In [23]:
# ============================================================
# CELL 11 — Save manifest and splits
# ============================================================

manifest.to_csv(MANIFEST_OUT, index=False)
splits_final.to_csv(SPLITS_OUT, index=False)

print("=" * 100)
print("PHASE 3 OUTPUT SAVED")
print("=" * 100)
print("Manifest:", MANIFEST_OUT)
print("Manifest rows:", len(manifest))
print("Splits  :", SPLITS_OUT)
print("Split rows:", len(splits_final))
print("Subjects:", sorted(manifest["subject_id"].unique().astype(int).tolist()))

# Save text report
report_path = DATASET_DIR / "dataset_analysis_report.txt"

with open(report_path, "w") as f:
    f.write("BioRob Phase 3 Dataset Analysis Report\n")
    f.write("======================================\n\n")
    f.write(f"ROOT_DIR: {ROOT_DIR}\n")
    f.write(f"LABELONLY_SUBPATH: {LABELONLY_SUBPATH}\n")
    f.write(f"Total valid labelonly files: {len(manifest)}\n")
    f.write(f"Total subjects: {manifest['subject_id'].nunique()}\n")
    f.write(f"Subjects: {sorted(manifest['subject_id'].unique().astype(int).tolist())}\n")
    f.write(f"Total windows: {len(splits_final)}\n")
    f.write(f"Window length (s): {SLIDE_WIN_S}\n")
    f.write(f"Window stride (s): {SLIDE_STRIDE_S}\n")
    f.write(f"Onset anchor enabled: {ENABLE_ONSET_ANCHOR_WINDOWS}\n")
    f.write(f"Phase3 train balancing enabled: {BALANCE_TRAIN_WINDOWS_IN_PHASE3}\n\n")

    f.write("Important exclusion note:\n")
    f.write("Phase 3 reads only Phase 2B labelonly files. Trials without successful Phase 1C synchronization do not have labelonly outputs and are therefore excluded automatically from Phase 3 and later phases.\n\n")

    f.write("Files by subject:\n")
    for sid, n in manifest.groupby("subject_id")["file"].count().items():
        f.write(f"  Sub-{int(sid)}: {int(n)} files\n")

    f.write("\nWindow label_action distribution:\n")
    for label, n in splits_final["label_action"].value_counts().sort_index().items():
        f.write(f"  {label}: {int(n)} windows\n")

    f.write("\nWindow task_target distribution:\n")
    for label, n in splits_final["task_target"].value_counts().sort_index().items():
        f.write(f"  {label}: {int(n)} windows\n")

    f.write("\nFold/split summary:\n")
    f.write(fold_summary.to_string(index=False))
    f.write("\n")

print("Report:", report_path)

# Final file existence check
assert MANIFEST_OUT.exists(), "Manifest was not saved."
assert SPLITS_OUT.exists(), "Splits were not saved."

print("\n✅ PHASE 3 COMPLETED SUCCESSFULLY.")
print("Use these outputs in Phase 4/Phase 5:")
print(" -", MANIFEST_OUT)
print(" -", SPLITS_OUT)

PHASE 3 OUTPUT SAVED
Manifest: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/manifest_v1.csv
Manifest rows: 1552
Splits  : /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/splits_v1.csv
Split rows: 475798
Subjects: [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 19, 20, 21, 22, 23]
Report: /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/dataset_analysis_report.txt

✅ PHASE 3 COMPLETED SUCCESSFULLY.
Use these outputs in Phase 4/Phase 5:
 - /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/manifest_v1.csv
 - /home/tsultan1/BioRob/Human Subject Data/_dataset_icml_v1/splits_v1.csv


In [24]:
# ============================================================
# CELL 12 — Final quick read-back check
# ============================================================

manifest_check = pd.read_csv(MANIFEST_OUT)
splits_check = pd.read_csv(SPLITS_OUT)

print("Manifest shape:", manifest_check.shape)
print("Splits shape  :", splits_check.shape)

print("\nManifest columns:")
print(list(manifest_check.columns))

print("\nSplits columns:")
print(list(splits_check.columns))

print("\nSplit counts:")
print(splits_check["split"].value_counts(dropna=False))

print("\nWindow type counts:")
print(splits_check["type"].value_counts(dropna=False))

print("\nLabel_action counts:")
print(splits_check["label_action"].value_counts(dropna=False).sort_index())

print("\nTask_target counts:")
print(splits_check["task_target"].value_counts(dropna=False).sort_index())

print("\n✅ Read-back check complete.")

Manifest shape: (1552, 7)
Splits shape  : (475798, 11)

Manifest columns:
['file', 'subject_id', 'fs_hz', 'duration_s', 'task_code', 'trial_id', 'fold_id']

Splits columns:
['file', 'subject_id', 'split', 'type', 'task_target', 'label_action', 'start_idx', 'end_idx', 'task_code', 'trial_id', 'fold_id']

Split counts:
split
train    411025
val       39731
test      25042
Name: count, dtype: int64

Window type counts:
type
sliding    475798
Name: count, dtype: int64

Label_action counts:
label_action
0     48222
1    427576
Name: count, dtype: int64

Task_target counts:
task_target
0     48222
1     90478
2    114627
3     78831
4     78603
5     65037
Name: count, dtype: int64

✅ Read-back check complete.
